In [ ]:
from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# ??? ??
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# ???? 1D ??? ??
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

# ??? ???
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# ?? ???? ??/???? ??
x_train_split, x_val_split, y_train_split, y_val_split = train_test_split(
    x_train_scaled, y_train, test_size=0.2, random_state=42
)

## 1.SVM

In [ ]:
# PCA? ?? ?? (??: ?? ??? ?? ??)
from sklearn.decomposition import PCA
pca = PCA(n_components=50)  # 50???? ??
x_train_pca = pca.fit_transform(x_train_scaled)
x_test_pca = pca.transform(x_test_scaled)

# 2. SVM ?? ??
from sklearn.svm import SVC
svm_model = SVC(kernel='rbf', C=1, gamma='scale')  # RBF ?? ??
svm_model.fit(x_train_pca, y_train)

# 3. ?? ??
y_pred = svm_model.predict(x_test_pca)

# ??? ? ?? ??? ??
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 2. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# ?? ??
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(x_train_split, y_train_split)

# ?? ? ??
y_pred_dt = decision_tree.predict(x_test_scaled)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Classification Report:\n", classification_report(y_test, y_pred_dt))

## 3. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ?? ??
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest.fit(x_train_split, y_train_split)

# ?? ? ??
y_pred_rf = random_forest.predict(x_test_scaled)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

## 4. XGBoost

In [ ]:
from xgboost import XGBClassifier

# ?? ??
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(x_train_split, y_train_split, eval_set=[(x_val_split, y_val_split)], verbose=False)

# ?? ? ??
y_pred_xgb = xgb_model.predict(x_test_scaled)
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

## 5. LightGBM

In [ ]:
from lightgbm import LGBMClassifier
from lightgbm import log_evaluation

# ?? ??
lgb_model = LGBMClassifier(random_state=42)
lgb_model.fit(
    x_train_split, y_train_split,
    eval_set=[(x_val_split, y_val_split)],  # ?? ??? ??
    eval_metric='multi_logloss',           # ?? ??? ?? ??
    callbacks=[log_evaluation(period=1)]   # ?? ?? ?? ??? ??
)

# ?? ? ??
y_pred_lgb = lgb_model.predict(x_test_scaled)
print("LightGBM Accuracy:", accuracy_score(y_test, y_pred_lgb))
print("Classification Report:\n", classification_report(y_test, y_pred_lgb))